## Generate Data

In [0]:
from pyspark.sql.functions import *

df = (
    spark
    .range(0, 60, 1,1)
    .select(
        'id',
        (col('id') % 1000).alias('device_id'),
        (rand() * 100).alias('temperature_F')
    )
)
df.display()
df.write.mode('overwrite').saveAsTable('device_data')


## Computattionally Expansice UDF

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

@udf("double")
def F_to_Celsius(f):
    # Let's pretend some fancy math takes one second per row
    time.sleep(1)
    return (f -32) * (5/9)

celsius_df = (spark.table("device_data")
              .withColumn("celcsius", F_to_Celsius(col('temperature_F')))
              )

celsius_df.write.mode('overwrite').saveAsTable('celsius')

That took approximately one minute, which is kind of surprisong since we have about 60 seconds worth of computation, spread across multiple cores. Shoudn't it take significantly less time?

In [0]:
num_cores=4

@udf("double")
def F_to_Celsius(f):
    # Let's pretend some fancy math takes one second per row
    time.sleep(1)
    return (f -32) * (5/9)

celsius_df = (spark.table("device_data")
              .repartition(num_cores)
              .withColumn("celcsius", F_to_Celsius(col('temperature_F')))
              )

celsius_df.write.mode('overwrite').saveAsTable('celsius')

In [0]:
celsius_df.explain()

### SQL UDFs

The ability to create user-defined functions in Python and Scala is convenient since it allows you to extend functionality in the language of your choice. As far as optimization is concerned, however, it's important to know that SQL is generally the best choice, for a couple of reasons:  

    - SQL UDFs require less data serialization

    - Catalyst optimizer can operate within SQL UDFs

Let's see this in action now by comparing the performance of a SQL UDF to its Python counterpart.

First let's redefine the Python UDF from before, this time without the delay, so we can compare raw performance.

In [0]:
%sql
drop function if exists farh_to_cels;

create function farh_to_cels(farh double) returns double
return (farh - 32) * 5 / 9;

CREATE OR REPLACE TABLE celsius_sql AS
select farh_to_cels(temperature_F) as Farh_to_cels_convert from device_data;

In [0]:
%sql
select * from celsius_sql;

In [0]:
# Ver el contenido del conteo de filas afectadas
_sqldf.show()

# O si quieres usar funciones de PySpark con el resultado del CTAS
filas_insertadas = _sqldf.select("Farh_to_cels_convert").collect()[0][0]
print(f"Se han insertado {filas_insertadas} registros en la tabla celsius_sql")

In [0]:
%sql
explain select farh_to_cels(temperature_F ) as Farh_to_cels_convert from device_data

In [0]:
_sqldf.explain(True)